# Assignment 20 – Building & Deploying a Recommendation System

## PART 1 – Data Preprocessing

### Task 1: Load & Understand Dataset

Import the libraries.

In [ ]:
! pip install pandas numpy scikit-learn streamlit

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("tmdb_5000_movies.csv")

# Display dataset shape
print("Dataset Shape:", df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

# Display first 5 rows
print("\nFirst 5 Rows:")
print(df.head())

# Display dataset information
print("\nDataset Information:")
print(df.info())

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate values
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Display statistical information
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
import pandas as pd

df = pd.read_csv("tmdb_5000_movies.csv")

print("Column Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# Identify the columns used for recommendations.

print("Movie Name Column: title")
print("Text Feature Column: overview")

print("\nSample Movie Titles:")
print(df["title"].head())

print("\nSample Movie Overviews:")


## Task 2: Text Preprocessing for Recommendation

In [ ]:

import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Clean column names
df.columns = df.columns.astype(str).str.strip()

# Find text columns
text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Available Text Columns:", text_columns)

# Automatically select a text column
preferred = ["overview", "description", "tags", "genre", "genres"]

text_column = next(
    (col for col in preferred if col in df.columns),
    None
)

if text_column is None:
    if not text_columns:
        raise ValueError("No text column found in dataset.")
    text_column = max(
        text_columns,
        key=lambda col: df[col].fillna("").astype(str).str.len().mean()
    )

print("Selected Text Column:", text_column)

# Handle missing values
df[text_column] = df[text_column].fillna("").astype(str)

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
    ]

    return " ".join(words)

# Create clean_text column
df["clean_text"] = df[text_column].apply(clean_text)

# Display results
print("\nOriginal Text:")
print(df[text_column].head())

print("\nCleaned Text:")
print(df["clean_text"].head())

print("\nMissing Values:", df["clean_text"].isnull().sum())

print("\nTask 2 Completed Successfully!")


## PART 2 – Text Vectorization

### Task 3: Vectorization using TF-IDF

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Handle missing values
df["clean_text"] = df["clean_text"].fillna("").astype(str)

# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert text into numerical vectors
tfidf_matrix = tfidf.fit_transform(df["clean_text"])

# Display matrix shape
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

# Display first 20 features
print("\nFirst 20 Features:")
print(tfidf.get_feature_names_out()[:20])

# Display first 5 rows
print("\nFirst 5 Rows:")
print(tfidf_matrix[:5].toarray())

print("\nTask 3 Completed Successfully!")


### Task 4: Similarity Computation

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity
similarity = cosine_similarity(tfidf_matrix)

# Store similarity matrix
similarity_matrix = similarity

# Display matrix shape
print("Similarity Matrix Shape:")
print(similarity_matrix.shape)

# Display first 5 rows and columns
print("\nFirst 5 Similarity Scores:")
print(similarity_matrix[:5, :5])

# Check similarity between first two movies
if similarity_matrix.shape[0] > 1:
    print("\nSimilarity Between First Two Movies:")
    print(similarity_matrix[0][1])

print("\nTask 4 Completed Successfully!")


## PART 3 — Recommendation Logic

### Task 5: Build Recommendation Function

In [ ]:

# Task 5: Build Recommendation Function

def recommend(item_name, top_n=5):

    # Find movie index
    movie_indices = df[
        df["title"].str.lower().str.strip()
        == item_name.lower().strip()
    ].index

    if len(movie_indices) == 0:
        print("Movie not found!")
        return

    movie_index = movie_indices[0]

    # Get similarity scores
    scores = list(enumerate(similarity[movie_index]))

    # Sort scores in descending order
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Remove selected movie
    scores = [
        item for item in scores
        if item[0] != movie_index
    ]

    # Get top 5 recommendations
    scores = scores[:top_n]

    # Display recommendations
    recommendations = []

    for index, score in scores:
        recommendations.append({
            "Movie Name": df.iloc[index]["title"],
            "Similarity Score": round(score, 3)
        })

    return pd.DataFrame(recommendations)


# Test 1
print("Recommendations for:", df["title"].iloc[0])
display(recommend(df["title"].iloc[0], 5))

# Test 2
print("Recommendations for:", df["title"].iloc[1])
display(recommend(df["title"].iloc[1], 5))

# Test 3
print("Recommendations for:", df["title"].iloc[2])
display(recommend(df["title"].iloc[2], 5))


## PART 4 — Simple App Interface

### Task 6: Build UI using Streamlit

Objective: Create a simple web application where users can select a movie and receive recommendations.

1. Movie Recommendation System heading.
2. Movie selection dropdown.
3. Get Recommendations button.
4. List of five recommended movies.
5. Similarity scores.

## PART 5 — Version Control with Git & GitHub

### Task 7: Git & GitHub Setup


# STEP 1: Check Git
git --version

# STEP 2: Configure username
git config --global user.name "memanbeck121-web"

# STEP 3: Configure email
git config --global user.email "memanbeck@gmail.com"

# STEP 4: Initialize repository
git init

# STEP 5: Add all files
git add .

# STEP 6: Commit files
git commit -m "First commit"

# STEP 7: Rename branch
git branch -M main

# STEP 8: Connect GitHub repository
git remote add origin https://github.com/memanbeck121-web/Movie-Recommendation-System.git

# STEP 9: Upload files
git push -u origin main

# STEP 10: Check status
git status


## PART 6 — Deployment on Render

### Task 8: Deploy Application on Render 
### Task 9: Final Validation


import os
import subprocess
import time

# =====================================
# CHANGE THESE DETAILS
# =====================================

GITHUB_USERNAME = "memanbeck121-web"
GITHUB_REPO = "Movie-Recommendation-System"
YOUR_NAME = "Meman Beck"
YOUR_EMAIL = "memanbeck121@gmail.com"

GITHUB_URL = (
    f"https://github.com/memanbeck121-web/Movie-Recommendation-System.git"
)

# =====================================
# CHECK PROJECT FILES
# =====================================

print("Checking project files...")

required_files = ["app.py", "tmdb_5000_movies.csv"]

for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"{file} not found! Keep it in the project folder.")

print("All required files found!")

# =====================================
# CREATE requirements.txt
# =====================================

requirements = """pandas
numpy
scikit-learn
streamlit
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created!")

# =====================================
# CREATE README.md
# =====================================

readme = f"""# Movie Recommendation System

## Project Description
A content-based movie recommendation system using TF-IDF
and cosine similarity.

## Dataset
TMDB 5000 Movie Dataset

Kaggle:
https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Technologies
- Python
- Pandas
- Scikit-learn
- Streamlit

## Run Locally

Install requirements:

pip install -r requirements.txt

Run application:

streamlit run app.py

## GitHub Repository
https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}

## Live Application
Deployment URL will be added after deploying on Render.
"""

with open("README.md", "w") as file:
    file.write(readme)

print("README.md created!")

# =====================================
# GIT CONFIGURATION
# =====================================

subprocess.run(["git", "config", "--global", "user.name", YOUR_NAME], check=True)
subprocess.run(["git", "config", "--global", "user.email", YOUR_EMAIL], check=True)

# =====================================
# INITIALIZE GIT
# =====================================

if not os.path.exists(".git"):
    subprocess.run(["git", "init"], check=True)

subprocess.run(["git", "branch", "-M", "main"], check=True)

# =====================================
# CONNECT GITHUB
# =====================================

remotes = subprocess.run(
    ["git", "remote"],
    capture_output=True,
    text=True,
    check=True
).stdout.split()

if "origin" in remotes:
    subprocess.run(
        ["git", "remote", "set-url", "origin", GITHUB_URL],
        check=True
    )
else:
    subprocess.run(
        ["git", "remote", "add", "origin", GITHUB_URL],
        check=True
    )

# =====================================
# PUSH PROJECT TO GITHUB
# =====================================

subprocess.run(["git", "add", "."], check=True)

commit = subprocess.run(
    ["git", "commit", "-m", "Upload recommendation system"],
    capture_output=True,
    text=True
)

if commit.returncode == 0:
    print(commit.stdout)
else:
    print(commit.stdout)
    print(commit.stderr)

subprocess.run(["git", "push", "-u", "origin", "main"], check=True)

print("\nProject successfully pushed to GitHub!")

# =====================================
# TASK 8 - DEPLOY ON RENDER
# =====================================

print("\nTASK 8: DEPLOYMENT ON RENDER")
print("1. Open https://render.com/")
print("2. Sign in using your GitHub account.")
print("3. Click New + and select Web Service.")
print("4. Connect your Movie-Recommendation-System repository.")
print("5. Select Python as the runtime.")

print("\nUse these Render settings:")
print("Build Command: pip install -r requirements.txt")
print("Start Command: streamlit run app.py --server.address 0.0.0.0 --server.port $PORT")

print("\nNow deploy your application on Render.")
print("Wait until the service shows Live.")

# =====================================
# TASK 9 - ADD DEPLOYED URL
# =====================================

DEPLOYED_URL = input(
    "\nAfter deployment, paste your actual Render URL here: "
).strip()

if not DEPLOYED_URL.startswith("https://"):
    raise ValueError("Please enter a valid HTTPS Render URL.")

with open("README.md", "r") as file:
    readme = file.read()

readme = readme.replace(
    "Deployment URL will be added after deploying on Render.",
    DEPLOYED_URL
)

with open("README.md", "w") as file:
    file.write(readme)

# =====================================
# UPDATE GITHUB
# =====================================

subprocess.run(["git", "add", "README.md"], check=True)

subprocess.run(
    ["git", "commit", "-m", "Add deployed Render URL"],
    check=True
)

subprocess.run(["git", "push"], check=True)

print("\nTASK 8 AND TASK 9 COMPLETED!")
print("GitHub:", f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}")
print("Render:", DEPLOYED_URL)
